<a href="https://colab.research.google.com/github/Joe-Something/AAI2026/blob/main/Ex2_CodeGenReACT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# Exercise 2: Code Generation with ReACT Prompting
# Environment: Google Colab
# Library: google-genai, pandas, matplotlib
# ==============================================================================

!pip install -q google-genai

import io
import sys
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from google import genai
from google.genai import errors
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

PRIMARY_MODEL = "gemini-3.6-flash"

def safe_generate_content(prompt, system_instruction=None):
    config = None
    if system_instruction:
        config = genai.types.GenerateContentConfig(system_instruction=system_instruction)

    for attempt in range(5):
        try:
            return client.models.generate_content(
                model=PRIMARY_MODEL,
                contents=prompt,
                config=config
            )
        except errors.ServerError:
            time.sleep(2 ** attempt)
        except errors.APIError as e:
            if e.code == 429:
                wait_time = 20 + (attempt * 5)
                print(f"[Rate Limit 429] Waiting {wait_time}s before retry (attempt {attempt + 1}/5)...")
                time.sleep(wait_time)
            else:
                raise RuntimeError(f"API Error ({e.code}): {e.message}")
        except Exception as e:
            if getattr(e, 'code', None) == 429 or "429" in str(e):
                wait_time = 20 + (attempt * 5)
                print(f"[Rate Limit 429] Waiting {wait_time}s before retry (attempt {attempt + 1}/5)...")
                time.sleep(wait_time)
            else:
                raise RuntimeError(f"Unexpected error: {e}")

    raise RuntimeError(f"Failed to generate content with model {PRIMARY_MODEL} after retries.")

sample_csv_data = """CustomerID,CustomerName,Region,SalesAmount,Date
C101,Acme Corp,North,12500.50,2026-01-15
C102,Beta LLC,South,8400.00,2026-01-18
C103,Gamma Inc,North,15300.75,2026-02-01
C104,Delta Co,East,4200.00,2026-02-10
C105,Epsilon Ltd,West,19800.00,2026-02-14
C106,Acme Corp,North,7600.00,2026-03-01
"""

with open("sales_data.csv", "w") as f:
    f.write(sample_csv_data)

def execute_python_code(code_str: str):
    old_stdout = sys.stdout
    redirected_output = sys.stdout = io.StringIO()
    error = None
    try:
        exec_globals = {"pd": pd, "plt": plt}
        exec(code_str, exec_globals)
    except Exception as e:
        error = f"{type(e).__name__}: {str(e)}"
    finally:
        sys.stdout = old_stdout

    return redirected_output.getvalue(), error

def run_react_loop(task_description: str, max_turns: int = 3):
    react_system_instruction = """
    You are an expert Python Data Analyst operating in a ReACT (Reasoning + Acting) loop.
    You will solve data processing tasks by iterating through Thought, Action, and Observation stages.

    STRICT FORMAT REQUIREMENT:
    You must format your responses using the following blocks:

    Thought: <Your step-by-step reasoning and strategy>
    Action:
    ```python
    # Executable Python code here
    ```

    Rule: Only output ONE Thought and ONE Action code block per turn.
    Allowed Libraries: pandas, matplotlib, io, sys.
    """

    user_prompt = f"Task: {task_description}"
    print(f"=== STARTING ReACT LOOP FOR TASK: {task_description} ===\n")

    current_input = user_prompt

    for turn in range(1, max_turns + 1):
        print(f"--- TURN {turn} ---")

        response = safe_generate_content(
            prompt=current_input,
            system_instruction=react_system_instruction
        )

        response_text = response.text
        print(response_text)

        if "```python" in response_text:
            code_block = response_text.split("```python")[1].split("```")[0].strip()
        elif "```" in response_text:
            code_block = response_text.split("```")[1].split("```")[0].strip()
        else:
            print("No executable code block found.")
            break

        stdout_val, error_val = execute_python_code(code_block)

        if error_val:
            observation = f"EXECUTION ERROR:\n{error_val}"
            print(f"\nObservation:\n{observation}\n")
        else:
            observation = f"SUCCESSFUL EXECUTION:\nOutput:\n{stdout_val}"
            print(f"\nObservation:\n{observation}\n")
            print("ReACT Task completed successfully!")
            break

        current_input = f"{response_text}\n\nObservation:\n{observation}\n\nFix the error and provide the updated Thought and Action block."
        time.sleep(12)  # Pace between turns for RPM limits

task = """
Read 'sales_data.csv'. Calculate:
1. Total overall sales.
2. The customer with the highest total spend across all transactions.
3. Plot a bar chart showing SalesAmount by Region using matplotlib and save it as 'sales_by_region.png'.
Print the results clearly to stdout.
"""

run_react_loop(task)

=== STARTING ReACT LOOP FOR TASK: 
Read 'sales_data.csv'. Calculate:
1. Total overall sales.
2. The customer with the highest total spend across all transactions.
3. Plot a bar chart showing SalesAmount by Region using matplotlib and save it as 'sales_by_region.png'.
Print the results clearly to stdout.
 ===

--- TURN 1 ---
Thought:
Let's inspect the first few rows and column names of `sales_data.csv` to understand its structure and data types.

Action:
```python
import pandas as pd

df = pd.read_csv('sales_data.csv')
print(df.info())
print(df.head())
```

Observation:
SUCCESSFUL EXECUTION:
Output:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   CustomerID    6 non-null      object 
 1   CustomerName  6 non-null      object 
 2   Region        6 non-null      object 
 3   SalesAmount   6 non-null      float64
 4   Date          6 non-null      o